# StarV Verification：3 个环境后台验证

环境：`cartpole` / `mountain_car` / `pendulum`

每个 case 都会使用对应的 `config/starv_verification/<env>.json`，通过 MPI 在后台运行 `verify.py`。默认启动 **128 个 MPI rank**，并把每个 rank 的 OpenBLAS、OMP 和 Gurobi 线程数限制为 1。

**运行前**：Jupyter kernel 切换成 `starv_shared`（`/home/tealab_shared/starv/env/starv_shared`）。本文件放在 `verifiable_wm/notebooks/` 下，第一个 code cell 会自动定位仓库根目录并切换过去。

日志写入 `log/<env>_verify.out`。三个 case 相互独立；不要一次全部启动，否则会同时占用 384 个 MPI rank。任务通过新 session 脱离 notebook，关闭页面或断开 kernel 后仍会继续运行。kernel 重启会丢失本页记录的 PID，可在仓库根目录运行：

```bash
pgrep -af 'verify.py config/starv_verification'
```

In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "verify.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
assert (NOTEBOOK_DIR / "verify.py").exists(), (
    "没找到仓库根目录（verify.py），请从 verifiable_wm/ 或 verifiable_wm/notebooks/ 启动 Jupyter"
)
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print("repo root:", NOTEBOOK_DIR)

In [ ]:
import os
import subprocess
from pathlib import Path

MPIRUN = Path("/home/tealab_shared/starv/env/starv_shared/bin/mpirun")
STARV_PYTHON = Path("/home/tealab_shared/starv/env/starv_shared/bin/python")
SUPPORTED_ENVS = ("cartpole", "mountain_car", "pendulum")
VERIFICATION_PIDS = {}


def start_verification(env_name, nproc=128):
    if env_name not in SUPPORTED_ENVS:
        raise ValueError(f"不支持的环境: {env_name}; 可选值: {SUPPORTED_ENVS}")
    if not isinstance(nproc, int) or isinstance(nproc, bool) or nproc <= 0:
        raise ValueError("nproc 必须是正整数")

    config_rel = Path("config") / "starv_verification" / f"{env_name}.json"
    config_path = NOTEBOOK_DIR / config_rel
    for label, path in (
        ("config", config_path),
        ("mpirun", MPIRUN),
        ("python", STARV_PYTHON),
    ):
        if not path.is_file():
            raise FileNotFoundError(f"找不到 {label}: {path}")
    if not os.access(MPIRUN, os.X_OK):
        raise PermissionError(f"mpirun 不可执行: {MPIRUN}")
    if not os.access(STARV_PYTHON, os.X_OK):
        raise PermissionError(f"python 不可执行: {STARV_PYTHON}")

    log_dir = NOTEBOOK_DIR / "log"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{env_name}_verify.out"
    env = os.environ.copy()
    env.update(
        OPENBLAS_NUM_THREADS="1",
        OMP_NUM_THREADS="1",
        GRB_THREADS="1",
    )
    command = [
        str(MPIRUN),
        "-np",
        str(nproc),
        str(STARV_PYTHON),
        "verify.py",
        config_rel.as_posix(),
    ]

    with log_path.open("wb") as log_file:
        process = subprocess.Popen(
            command,
            cwd=NOTEBOOK_DIR,
            env=env,
            stdin=subprocess.DEVNULL,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    VERIFICATION_PIDS[env_name] = process.pid
    print(f"[Started] env={env_name}, ranks={nproc}, PID={process.pid}")
    print(f"[Log] {log_path}")
    print(f"[Watch] tail -f {log_path.relative_to(NOTEBOOK_DIR)}")
    print(f"[Stop] kill {process.pid}  # 执行后请再次检查 MPI 子进程")
    return process.pid


def _pid_is_running(pid):
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return False
    except PermissionError:
        return True
    return True


def show_verification_status():
    if not VERIFICATION_PIDS:
        print("当前 kernel 尚未启动 verification；kernel 重启后请使用 pgrep 查看后台任务。")
        return
    for env_name, pid in VERIFICATION_PIDS.items():
        status = "running" if _pid_is_running(pid) else "finished"
        print(f"{env_name}: PID={pid}, status={status}")


def show_log(env_name, lines=20):
    if env_name not in SUPPORTED_ENVS:
        raise ValueError(f"不支持的环境: {env_name}; 可选值: {SUPPORTED_ENVS}")
    if not isinstance(lines, int) or isinstance(lines, bool) or lines <= 0:
        raise ValueError("lines 必须是正整数")
    log_path = NOTEBOOK_DIR / "log" / f"{env_name}_verify.out"
    if not log_path.is_file():
        print(f"日志尚不存在: {log_path}")
        return
    result = subprocess.run(
        ["tail", "-n", str(lines), str(log_path)],
        check=False,
        text=True,
        capture_output=True,
    )
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")

## Case 1: CartPole

使用 `config/starv_verification/cartpole.json`，日志写入 `log/cartpole_verify.out`。运行后本 cell 会立即返回 PID；verification 在后台继续。

In [ ]:
start_verification("cartpole")

## Case 2: MountainCar

使用 `config/starv_verification/mountain_car.json`，日志写入 `log/mountain_car_verify.out`。如果另一个 128-rank 任务仍在运行，请先确认机器还有足够资源。

In [ ]:
start_verification("mountain_car")

## Case 3: Pendulum

使用 `config/starv_verification/pendulum.json`，日志写入 `log/pendulum_verify.out`。如果另一个 128-rank 任务仍在运行，请先确认机器还有足够资源。

In [ ]:
start_verification("pendulum")

## 查看当前 kernel 启动的任务

这里只检查内存中记录的 launcher PID，不会启动或停止任务。kernel 重启后请改用开头的 `pgrep` 命令。

In [ ]:
show_verification_status()

## 查看日志末尾

默认显示 CartPole 日志的最后 20 行。需要持续跟踪时，复制启动 cell 打印的 `tail -f` 命令到终端执行。

In [ ]:
show_log("cartpole", lines=20)
# show_log("mountain_car", lines=20)
# show_log("pendulum", lines=20)